# Run Explorer
Browse trained models under `results/`, compare runs for a given product
(category), and reproduce a past run's exact configuration.

Uses `run_registry.py` — a standalone module that only reads paths/YAML,
so none of this requires loading a model, dataset, or the fiftyone/anomalib
stack.

## 1. Setup

In [ ]:
import sys

sys.path.append("src")

from pathlib import Path
import pandas as pd

from src.run_registry import (
    list_runs,
    best_run,
    reproduce_run,
    find_existing_runs_with_config,
    RunSummary,
)

resultsDir = Path("results")

## 2. List all runs

`list_runs` scans every `manifest.yaml` under `results/**/runs/*/`, so this
works regardless of which dataset/category/model produced each run.

In [ ]:
runs = list_runs(resultsDir)

print(f"Found {len(runs)} run(s):\n")
for r in runs:
    print(r)

## 3. View runs as a table

Handy for scanning many runs at once, sorting by metric, or exporting.

In [ ]:
def runs_to_dataframe(runs: list[RunSummary]) -> pd.DataFrame:
    rows = []
    for r in runs:
        row = {
            "runId": r.runId,
            "model": r.modelName,
            "dataset": r.datasetName,
            "category": r.category,
            "complete": r.isComplete,
            "createdAt": r.createdAt,
            "configHash": r.configHash,
        }
        row.update(r.metrics)
        rows.append(row)
    return pd.DataFrame(rows)

df = runs_to_dataframe(runs)
df

## 4. Filter to a specific product (category)

This is the core "which models have I trained for this product" view.

In [ ]:
productName = "cable"

product_runs = list_runs(resultsDir, productName=productName)

print(f"{len(product_runs)} run(s) for product '{productName}':\n")
for r in product_runs:
    print(r)

## 5. Find the best run for a product by metric

Only considers **complete** runs (all tile checkpoints present) — an
incomplete run is never a valid candidate, regardless of what its logged
metrics say.

In [ ]:
metricName = "AUROC"

best = best_run(resultsDir, productName=productName, metric=metricName, mode="max")

if best is None:
    print(f"No complete run with metric '{metricName}' found for product '{productName}'.")
else:
    print(f"Best run for '{productName}' by {metricName}:")
    print(best)
    print(f"\nCheckpoints: {best.runDir / 'checkpoints'}")

## 6. Check for duplicate configurations

Before kicking off a new training run, see if an identical configuration has
already been tried — useful to avoid accidentally retraining the exact same
setup, or to confirm run-to-run variance when that's intentional.

In [ ]:
# Example: reuse an existing run's config_hash to see what else matches it.
if runs:
    exampleHash = runs[0].configHash
    duplicates = find_existing_runs_with_config(resultsDir, exampleHash)
    print(f"Runs matching config_hash={exampleHash}:")
    for d in duplicates:
        print(f"  {d}")
else:
    print("No runs available to demonstrate duplicate-config lookup.")

## 7. Reproduce a past run

`reproduce_run` rebuilds config objects from the run's own **frozen** config
copies (`runDir/configs/...`), not the live/shared config tree — so this
still works correctly even if the live configs have since changed.

This returns config objects only; it does not start training. Feed the
result into `manager.train(...)` with a fresh `runLabel` to actually rerun it
(which creates a *new*, separate run rather than overwriting the original).

In [ ]:
if best is not None:
    reproduced = reproduce_run(best.runDir)

    print("Reconstructed configs from:", best.runDir)
    print("Original config_hash:", reproduced["original_manifest"]["config_hash"])
    print("Original created_at:", reproduced["original_manifest"]["created_at"])
    print()
    print("modelConfig:", reproduced["modelConfig"])
else:
    print("No 'best' run selected above — pick a runDir manually to try reproduce_run(), e.g.:")
    print("  reproduce_run(Path('results/MVTecADShort/cable/Padim/tiled/runs/<run_id>'))")

## 8. Rerun the reproduced configuration (optional)

Uncomment to actually retrain using the reconstructed settings. This creates
a brand-new run directory (new `run_id`) rather than touching the original —
the relationship between the two is discoverable afterwards via
`find_existing_runs_with_config`, since both share the same `config_hash`.

In [ ]:
# from src.manager import AnomalyDetectionManager as ADM
# from src.manager import DatasetSession as DS
#
# manager = ADM(outputDir=resultsDir, configDir=Path("configs/"))
# manager.generateModel(modelConfig=reproduced["modelConfig"])
#
# datasetSession = DS.loadDatasetFromDatabase(reproduced["original_manifest"]["config"]["dataset"]["name"])
# datasetSession.select_category(reproduced["original_manifest"]["config"]["dataset"]["category"])
#
# manager.train(
#     trainerConfig=reproduced["trainerConfig"],
#     modelConfig=reproduced["modelConfig"],
#     datamoduleConfig=reproduced["datamoduleConfig"],
#     tilingPipelineConfig=reproduced["tilingPipelineConfig"],
#     datasetSession=datasetSession,
#     runLabel="repro",
# )